# Phase 7: Validation-Based Threshold Selection

In Phase 6 we saw that changing the tumor-probability threshold could reduce false positives. In this phase we choose that threshold on the validation split, then apply the selected threshold once to the test split.

This keeps the test set as a final evaluation set instead of using it to tune the decision rule.

## 1. Setup

Load the simple CNN checkpoint, project utilities, and dataset split. Finds the project root, imports the trained model and evaluation helpers, and points to the Phase 3 checkpoint.

In [1]:
import csv
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from spectiqai.data import KidneyImageDataset, count_by_class, load_split_csv
from spectiqai.evaluation import binary_classification_metrics, collect_predictions
from spectiqai.models import SimpleKidneyCNN

DEVICE = torch.device("cpu")
RESULTS_DIR = PROJECT_ROOT / "results"
SPLIT_CSV_PATH = RESULTS_DIR / "dataset_splits.csv"
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "simple_cnn" / "simple_cnn_epoch_3.pt"

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {CHECKPOINT_PATH.exists()}")

Project root: c:\Users\smart\Coding World\SpectiqAI
Device: cpu
Checkpoint path: c:\Users\smart\Coding World\SpectiqAI\models\simple_cnn\simple_cnn_epoch_3.pt
Checkpoint exists: True


## 2. Load Validation and Test Data

Use validation data for threshold selection and test data for final reporting. Loads both splits with the same deterministic preprocessing used in Phase 4.

In [2]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0

splits = load_split_csv(SPLIT_CSV_PATH, project_root=PROJECT_ROOT)
val_records = splits["validation"]
test_records = splits["test"]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

val_dataset = KidneyImageDataset(val_records, transform=eval_transform)
test_dataset = KidneyImageDataset(test_records, transform=eval_transform)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Validation images: {len(val_dataset)} | counts: {count_by_class(val_records)}")
print(f"Test images: {len(test_dataset)} | counts: {count_by_class(test_records)}")

Validation images: 1500 | counts: {'Normal': 750, 'Tumor': 750}
Test images: 1500 | counts: {'Normal': 750, 'Tumor': 750}


## 3. Load the Simple CNN

Restore the best current baseline model. Rebuilds the simple CNN architecture and loads saved weights.

In [3]:
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

model = SimpleKidneyCNN(num_classes=2).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded checkpoint epoch: {checkpoint['epoch']}")
print(f"Checkpoint metrics: {checkpoint['metrics']}")

Loaded checkpoint epoch: 3
Checkpoint metrics: {'epoch': 3, 'train_loss': 0.07220970629528164, 'train_accuracy': 0.9754285714285714, 'val_loss': 0.025610201369971036, 'val_accuracy': 0.9886666666666667}


## 4. Collect Validation and Test Probabilities

Get tumor probabilities without changing model weights. Runs the model on validation and test loaders and stores labels plus probabilities.

In [4]:
val_true_labels, val_default_predictions, val_tumor_probabilities = collect_predictions(model, val_loader, DEVICE)
test_true_labels, test_default_predictions, test_tumor_probabilities = collect_predictions(model, test_loader, DEVICE)

print(f"Validation probabilities: {len(val_tumor_probabilities)}")
print(f"Test probabilities: {len(test_tumor_probabilities)}")
print(f"Default validation metrics: {binary_classification_metrics(val_true_labels, val_default_predictions)}")
print(f"Default test metrics: {binary_classification_metrics(test_true_labels, test_default_predictions)}")

Validation probabilities: 1500
Test probabilities: 1500
Default validation metrics: {'accuracy': 0.9886666666666667, 'precision': 0.9778357235984355, 'recall': 1.0, 'f1_score': 0.9887936717205009, 'true_positive': 750.0, 'true_negative': 733.0, 'false_positive': 17.0, 'false_negative': 0.0}
Default test metrics: {'accuracy': 0.9893333333333333, 'precision': 0.97911227154047, 'recall': 1.0, 'f1_score': 0.9894459102902375, 'true_positive': 750.0, 'true_negative': 734.0, 'false_positive': 16.0, 'false_negative': 0.0}


## 5. Choose Threshold on Validation Data

Select the threshold that maximizes validation F1 score. Evaluates thresholds from 0.01 to 0.99, sorts by validation F1, and saves the full threshold table.

In [5]:
def predictions_at_threshold(probabilities, threshold):
    return [1 if probability >= threshold else 0 for probability in probabilities]

validation_threshold_rows = []
for step in range(1, 100):
    threshold = step / 100
    predictions = predictions_at_threshold(val_tumor_probabilities, threshold)
    metrics = binary_classification_metrics(val_true_labels, predictions)
    validation_threshold_rows.append({"threshold": threshold, **metrics})

validation_threshold_rows = sorted(
    validation_threshold_rows,
    key=lambda row: (row["f1_score"], row["recall"], row["precision"]),
    reverse=True,
)
selected_threshold = validation_threshold_rows[0]["threshold"]

validation_threshold_path = RESULTS_DIR / "simple_cnn_validation_thresholds.csv"
with validation_threshold_path.open("w", newline="", encoding="utf-8") as csv_file:
    fieldnames = list(validation_threshold_rows[0].keys())
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(validation_threshold_rows)

print(f"Selected threshold: {selected_threshold:.2f}")
print("Top validation thresholds:")
for row in validation_threshold_rows[:10]:
    print(
        f"threshold={row['threshold']:.2f}: accuracy={row['accuracy']:.4f}, "
        f"precision={row['precision']:.4f}, recall={row['recall']:.4f}, f1={row['f1_score']:.4f}"
    )
print(f"Saved validation threshold table: {validation_threshold_path}")

Selected threshold: 0.85
Top validation thresholds:
threshold=0.85: accuracy=0.9987, precision=0.9973, recall=1.0000, f1=0.9987
threshold=0.92: accuracy=0.9987, precision=1.0000, recall=0.9973, f1=0.9987
threshold=0.81: accuracy=0.9980, precision=0.9960, recall=1.0000, f1=0.9980
threshold=0.82: accuracy=0.9980, precision=0.9960, recall=1.0000, f1=0.9980
threshold=0.83: accuracy=0.9980, precision=0.9960, recall=1.0000, f1=0.9980
threshold=0.84: accuracy=0.9980, precision=0.9960, recall=1.0000, f1=0.9980
threshold=0.86: accuracy=0.9980, precision=0.9973, recall=0.9987, f1=0.9980
threshold=0.87: accuracy=0.9980, precision=0.9973, recall=0.9987, f1=0.9980
threshold=0.88: accuracy=0.9980, precision=0.9973, recall=0.9987, f1=0.9980
threshold=0.89: accuracy=0.9980, precision=0.9973, recall=0.9987, f1=0.9980
Saved validation threshold table: c:\Users\smart\Coding World\SpectiqAI\results\simple_cnn_validation_thresholds.csv


## 6. Apply the Selected Threshold to the Test Set

Report final test metrics using the validation-selected threshold. Applies the selected threshold to test probabilities, computes final metrics, and saves predictions plus summary metrics.

In [6]:
test_selected_predictions = predictions_at_threshold(test_tumor_probabilities, selected_threshold)
test_selected_metrics = binary_classification_metrics(test_true_labels, test_selected_predictions)
test_selected_metrics = {"selected_threshold": selected_threshold, **test_selected_metrics}

metrics_path = RESULTS_DIR / "simple_cnn_validation_selected_threshold_test_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(test_selected_metrics.keys()))
    writer.writeheader()
    writer.writerow(test_selected_metrics)

predictions_path = RESULTS_DIR / "simple_cnn_validation_selected_threshold_test_predictions.csv"
with predictions_path.open("w", newline="", encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(["true_label", "default_prediction", "threshold_prediction", "tumor_probability", "selected_threshold"])
    for true_label, default_prediction, threshold_prediction, probability in zip(
        test_true_labels,
        test_default_predictions,
        test_selected_predictions,
        test_tumor_probabilities,
    ):
        writer.writerow([true_label, default_prediction, threshold_prediction, probability, selected_threshold])

print("Final test metrics with validation-selected threshold:")
for key, value in test_selected_metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")
print(f"Saved metrics: {metrics_path}")
print(f"Saved predictions: {predictions_path}")

Final test metrics with validation-selected threshold:
selected_threshold: 0.8500
accuracy: 0.9980
precision: 0.9973
recall: 0.9987
f1_score: 0.9980
true_positive: 749.0000
true_negative: 748.0000
false_positive: 2.0000
false_negative: 1.0000
Saved metrics: c:\Users\smart\Coding World\SpectiqAI\results\simple_cnn_validation_selected_threshold_test_metrics.csv
Saved predictions: c:\Users\smart\Coding World\SpectiqAI\results\simple_cnn_validation_selected_threshold_test_predictions.csv


## Phase 7 Completion Checklist

Verify that the following outputs have been generated:
- `simple_cnn_validation_thresholds.csv` ranks thresholds using validation data.
- `simple_cnn_validation_selected_threshold_test_metrics.csv` reports final test metrics.
- `simple_cnn_validation_selected_threshold_test_predictions.csv` stores the new test predictions.

The threshold has now been selected without directly tuning on the test set. Next step: document the final model choice and threshold, or run fine-tuning experiments if transfer learning still needs improvement.